# Modulo 5 · Lezione 5 — Autenticazione

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Perché autenticare? Tipi di autenticazione |
| 2 | API key — implementazione semplice |
| 3 | Hashing delle password con `bcrypt` |
| 4 | JWT — cosa sono e come funzionano |
| 5 | OAuth2 Password Flow con FastAPI |
| 6 | Ruoli e permessi |
| 7 | Proteggere gli endpoint con `Depends` |

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'python-jose[cryptography]', 'passlib[bcrypt]', 'python-multipart', 'requests', 'sqlalchemy']:
    try:
        __import__(pkg.split('[')[0].replace('-','_'))
        print(f"{pkg.split('[')[0]}: già installato")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: installato")

---
## 1. Tipi di autenticazione

```
AUTENTICAZIONE = verificare che sei chi dici di essere
AUTORIZZAZIONE = verificare che puoi fare ciò che stai tentando

Tipi comuni nelle API:
┌────────────────┬─────────────────────────────────┬────────────────────┐
│ Tipo           │ Come funziona                   │ Quando usarlo      │
├────────────────┼─────────────────────────────────┼────────────────────┤
│ API Key        │ Header X-API-Key: abc123        │ App M2M, script    │
│ Basic Auth     │ Authorization: Basic base64(..) │ Admin interno       │
│ Bearer Token   │ Authorization: Bearer <token>   │ La maggior parte delle API │
│ OAuth2         │ Flusso con login + token         │ API con utenti      │
│ JWT            │ Token firmato, autocontenuto     │ Microservizi        │
│ OAuth2+OIDC    │ Delega esterna (Google...)       │ Login social         │
└────────────────┴─────────────────────────────────┴────────────────────┘

In questo corso: API Key + JWT con OAuth2 Password Flow
```

---
## 2. API Key

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Security
from fastapi.security import APIKeyHeader
import secrets, uvicorn, threading, time, requests

app_key = FastAPI(title="API con API Keys")

# Archivio delle chiavi (in produzione: database)
API_KEYS = {
    "sk_live_abc123": {"nombre": "App mobile",   "permisos": ["read"]},
    "sk_live_xyz789": {"nombre": "Dashboard",    "permisos": ["read", "write"]},
    "sk_live_admin0": {"nombre": "Script admin", "permisos": ["read", "write", "delete"]},
}

# Schema: cerca l'header 'X-API-Key'
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)

def verificar_api_key(api_key: str = Security(api_key_header)):
    if not api_key or api_key not in API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="API key non valida o assente",
            headers={"WWW-Authenticate": "ApiKey"}
        )
    return API_KEYS[api_key]

def requiere_permiso(permiso: str):
    def verificar(cliente=Depends(verificar_api_key)):
        if permiso not in cliente["permisos"]:
            raise HTTPException(status_code=403, detail=f"Permesso '{permiso}' richiesto")
        return cliente
    return verificar


@app_key.get("/datos")  # solo lettura
def leer(cliente=Depends(verificar_api_key)):
    return {"datos": [1, 2, 3], "acceso": cliente["nombre"]}

@app_key.post("/datos")  # richiede 'write'
def escribir(cliente=Depends(requiere_permiso("write"))):
    return {"mensaje": f"Scritto da: {cliente['nombre']}"}

@app_key.delete("/datos")  # richiede 'delete'
def borrar(cliente=Depends(requiere_permiso("delete"))):
    return {"mensaje": "Eliminato"}

@app_key.get("/generar-key")  # genera una nuova chiave casuale
def generar():
    return {"key": f"sk_live_{secrets.token_urlsafe(16)}"}


cfg = uvicorn.Config(app_key, host="127.0.0.1", port=8020, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B = "http://127.0.0.1:8020"

print("Senza chiave (401):")
r = requests.get(f"{B}/datos")
print(f"  {r.status_code}:", r.json())

print("\nChiave di sola lettura — GET (200):")
r = requests.get(f"{B}/datos", headers={"X-API-Key": "sk_live_abc123"})
print(f"  {r.status_code}:", r.json())

print("\nChiave di sola lettura — POST (403):")
r = requests.post(f"{B}/datos", headers={"X-API-Key": "sk_live_abc123"})
print(f"  {r.status_code}:", r.json())

print("\nChiave admin — DELETE (200):")
r = requests.delete(f"{B}/datos", headers={"X-API-Key": "sk_live_admin0"})
print(f"  {r.status_code}:", r.json())

---
## 3. Hashing delle password

**Mai** salvare le password in chiaro. Usa un hash irreversibile con sale:

```
password_in_chiaro → [bcrypt/argon2] → hash nel DB

Per verificare:
password_in_chiaro + hash_nel_DB → [bcrypt.verify] → True/False

Caratteristiche di bcrypt:
✓ Volutamente lento (rende più difficili gli attacchi brute force)
✓ Sale casuale incluso nell'hash
✓ Fattore di costo regolabile
✗ Non usare MD5/SHA1 per le password (veloci, senza sale)
```

In [ ]:
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verificar_password(password: str, hashed: str) -> bool:
    return pwd_context.verify(password, hashed)


contraseña = "miPassword123!"
hashed     = hash_password(contraseña)

print(f"Password originale:   {contraseña}")
print(f"Hash bcrypt:          {hashed[:30]}...")  # 60 caratteri totali
print(f"Sono uguali? (testo)  {contraseña == hashed}")
print(f"Verifica? (bcrypt)    {verificar_password(contraseña, hashed)}")
print(f"Password errata?      {verificar_password('otraPassword', hashed)}")

# Due hash della stessa password sono DIVERSI (sale casuale)
h2 = hash_password(contraseña)
print(f"\nStesso testo, hash diverso: {hashed[:20]!r} vs {h2[:20]!r}")
print(f"Ma la verifica continua a funzionare: {verificar_password(contraseña, h2)}")

---
## 4. JSON Web Tokens (JWT)

```
Un JWT ha 3 parti separate da punti:

eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9   ← HEADER  (base64)
.eyJzdWIiOiJ1c3VhcmlvMSIsImV4cCI6MTY5...  ← PAYLOAD (base64)
.SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQ   ← SIGNATURE

Il PAYLOAD contiene i claim:
{
  "sub":  "usuario1",    ← subject (chi è)
  "exp":  1700000000,    ← expiry (quando scade)
  "roles": ["admin"],    ← dati propri
  "iat":  1699996400     ← issued at
}

Il server firma con la propria chiave segreta.
Chiunque può LEGGERE il payload (non è segreto).
Nessuno può MODIFICARLO senza invalidare la firma.

IMPORTANTE: non salvare dati sensibili nel payload.
```

In [ ]:
from jose import jwt, JWTError
from datetime import datetime, timedelta, timezone

CLAVE_SECRETA = "super-secreto-que-solo-conoce-el-servidor-min-32-chars"
ALGORITMO     = "HS256"
EXPIRACION_MINUTOS = 30

def crear_token(datos: dict, expira_en: int = EXPIRACION_MINUTOS) -> str:
    payload = datos.copy()
    payload["exp"] = datetime.now(timezone.utc) + timedelta(minutes=expira_en)
    return jwt.encode(payload, CLAVE_SECRETA, algorithm=ALGORITMO)

def decodificar_token(token: str) -> dict:
    try:
        return jwt.decode(token, CLAVE_SECRETA, algorithms=[ALGORITMO])
    except JWTError as e:
        raise HTTPException(status_code=401, detail=f"Token non valido: {e}")


# Creare il token
token = crear_token({"sub": "usuario42", "nombre": "Ana García", "rol": "admin"})
print("Token JWT:")
print(token)
print(f"\nLunghezza: {len(token)} caratteri")
print(f"Parti:     {len(token.split('.'))} (header.payload.signature)")

# Decodificare
payload = decodificar_token(token)
print("\nPayload decodificato:")
for k, v in payload.items():
    if k == 'exp':
        print(f"  {k}: {datetime.fromtimestamp(v).strftime('%H:%M:%S')} (scade tra ~30 min)")
    else:
        print(f"  {k}: {v}")

# Token manomesso → errore
print("\nToken manomesso:")
try:
    decodificar_token(token[:-5] + "XXXXX")
except HTTPException as e:
    print(f"  Errore (atteso): {e.detail[:50]}")

---
## 5. OAuth2 Password Flow + JWT con FastAPI

```
FLUSSO:

1. Client → POST /token  {username, password}
2. Server → verifica le credenziali → genera il JWT → restituisce {access_token}
3. Client → GET /me  Authorization: Bearer <JWT>
4. Server → decodifica il JWT → identifica l'utente → restituisce i dati

Il server non salva sessioni → stateless → scalabile
```

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, ConfigDict
from typing import Optional

app_auth = FastAPI(title="API con JWT")

# ── Modelli ──────────────────────────────────────────────────
class Usuario(BaseModel):
    username: str
    nombre:   str
    email:    str
    rol:      str = "usuario"
    activo:   bool = True

class Token(BaseModel):
    access_token: str
    token_type:   str = "bearer"

# ── DB utenti (in produzione: tabella in SQLite) ─────────────────
USUARIOS_DB = {
    "ana": {
        "username": "ana",
        "nombre":   "Ana García",
        "email":    "ana@empresa.com",
        "rol":      "admin",
        "activo":   True,
        "password_hash": hash_password("secretoAna"),
    },
    "carlos": {
        "username": "carlos",
        "nombre":   "Carlos López",
        "email":    "carlos@empresa.com",
        "rol":      "usuario",
        "activo":   True,
        "password_hash": hash_password("secretoCarlos"),
    },
    "inactivo": {
        "username": "inactivo",
        "nombre":   "Utente Disabilitato",
        "email":    "off@empresa.com",
        "rol":      "usuario",
        "activo":   False,
        "password_hash": hash_password("clave123"),
    },
}

# ── OAuth2 scheme ─────────────────────────────────────────────
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

def autenticar_usuario(username: str, password: str) -> Optional[dict]:
    usuario = USUARIOS_DB.get(username)
    if not usuario:
        return None
    if not verificar_password(password, usuario["password_hash"]):
        return None
    return usuario

def obtener_usuario_actual(token: str = Depends(oauth2_scheme)) -> Usuario:
    payload   = decodificar_token(token)
    username  = payload.get("sub")
    if not username or username not in USUARIOS_DB:
        raise HTTPException(status_code=401, detail="Token non valido")
    datos = USUARIOS_DB[username]
    if not datos["activo"]:
        raise HTTPException(status_code=403, detail="Account disattivato")
    return Usuario(**{k: v for k, v in datos.items() if k != "password_hash"})

def solo_admin(usuario: Usuario = Depends(obtener_usuario_actual)) -> Usuario:
    if usuario.rol != "admin":
        raise HTTPException(status_code=403, detail="È richiesto il ruolo admin")
    return usuario


# ── Endpoint ─────────────────────────────────────────────────
@app_auth.post("/token", response_model=Token)
def login(form: OAuth2PasswordRequestForm = Depends()):
    usuario = autenticar_usuario(form.username, form.password)
    if not usuario:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Utente o password errati",
            headers={"WWW-Authenticate": "Bearer"}
        )
    if not usuario["activo"]:
        raise HTTPException(status_code=403, detail="Account disattivato")

    token = crear_token({"sub": usuario["username"], "rol": usuario["rol"]})
    return Token(access_token=token)


@app_auth.get("/me", response_model=Usuario)
def mi_perfil(usuario: Usuario = Depends(obtener_usuario_actual)):
    return usuario

@app_auth.get("/recursos")
def recursos_publicos(usuario: Usuario = Depends(obtener_usuario_actual)):
    return {"mensaje": f"Ciao {usuario.nombre}, ecco le tue risorse"}

@app_auth.get("/admin/usuarios")
def listar_usuarios(admin: Usuario = Depends(solo_admin)):
    return [
        {k: v for k, v in u.items() if k != "password_hash"}
        for u in USUARIOS_DB.values()
    ]


cfg = uvicorn.Config(app_auth, host="127.0.0.1", port=8021, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)
B = "http://127.0.0.1:8021"

print("=== Login Ana (admin) ===")
r = requests.post(f"{B}/token", data={"username": "ana", "password": "secretoAna"})
token_ana = r.json()["access_token"]
print(f"  Token: {token_ana[:40]}...")

print("\n=== Login con password errata ===")
r = requests.post(f"{B}/token", data={"username": "ana", "password": "mal"})
print(f"  {r.status_code}:", r.json()["detail"])

print("\n=== GET /me ===")
r = requests.get(f"{B}/me", headers={"Authorization": f"Bearer {token_ana}"})
print(r.json())

print("\n=== Admin vede gli utenti ===")
r = requests.get(f"{B}/admin/usuarios", headers={"Authorization": f"Bearer {token_ana}"})
print(f"  {r.status_code}: {[u['username'] for u in r.json()]}")

print("\n=== Carlos (non admin) prova /admin/usuarios ===")
r_carlos = requests.post(f"{B}/token", data={"username": "carlos", "password": "secretoCarlos"})
token_carlos = r_carlos.json()["access_token"]
r = requests.get(f"{B}/admin/usuarios", headers={"Authorization": f"Bearer {token_carlos}"})
print(f"  {r.status_code}:", r.json()["detail"])

---
## 6. Registrazione degli utenti

In [ ]:
# Endpoint di registrazione — andrebbe aggiunto all'app precedente

from pydantic import BaseModel, Field, EmailStr

class RegistroRequest(BaseModel):
    username: str  = Field(min_length=3, max_length=50, pattern=r"^[a-z0-9_]+$")
    nombre:   str  = Field(min_length=1)
    email:    str  = Field(min_length=5)
    password: str  = Field(min_length=8)

@app_auth.post("/registro", response_model=Usuario, status_code=201)
def registrar(datos: RegistroRequest):
    if datos.username in USUARIOS_DB:
        raise HTTPException(status_code=409, detail="Il nome utente esiste già")

    if any(u["email"] == datos.email for u in USUARIOS_DB.values()):
        raise HTTPException(status_code=409, detail="L'email è già registrata")

    nuevo = {
        "username":      datos.username,
        "nombre":        datos.nombre,
        "email":         datos.email,
        "rol":           "usuario",
        "activo":        True,
        "password_hash": hash_password(datos.password),
    }
    USUARIOS_DB[datos.username] = nuevo
    return Usuario(**{k: v for k, v in nuevo.items() if k != "password_hash"})


# Provare la registrazione
print("=== Registrazione nuovo utente ===")
r = requests.post(f"{B}/registro", json={
    "username": "lucia",
    "nombre":   "Lucía Martínez",
    "email":    "lucia@empresa.com",
    "password": "miClaveSegura99"
})
print(f"  {r.status_code}:", r.json())

print("\n=== Lucía fa login ===")
r = requests.post(f"{B}/token", data={"username": "lucia", "password": "miClaveSegura99"})
print(f"  {r.status_code}: token ricevuto")

print("\n=== Username duplicato ===")
r = requests.post(f"{B}/registro", json={
    "username": "lucia", "nombre": "Otro", "email": "otro@x.com", "password": "12345678"
})
print(f"  {r.status_code}:", r.json()["detail"])

---
## Pratica

### Esercizio 1 — Refresh token

**Consegna:** Implementa un sistema di refresh token:

- Al login, restituisci un `access_token` (scade in 15 min) e un `refresh_token` (scade in 7 giorni)
- `POST /token/refresh` — riceve il refresh token e restituisce un nuovo access token
- Il refresh token viene invalidato dopo l'uso (memorizzalo in un set in memoria)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import secrets
from datetime import datetime, timedelta, timezone

# Archivio dei refresh token attivi: {token: username}
refresh_tokens: dict[str, str] = {}

class TokenPair(BaseModel):
    access_token:  str
    refresh_token: str
    token_type:    str = "bearer"

class RefreshRequest(BaseModel):
    refresh_token: str

@app_auth.post("/token/v2", response_model=TokenPair)
def login_v2(form: OAuth2PasswordRequestForm = Depends()):
    usuario = autenticar_usuario(form.username, form.password)
    if not usuario: raise HTTPException(401, "Credenziali errate")

    access  = crear_token({"sub": form.username}, expira_en=15)
    refresh = secrets.token_urlsafe(32)
    refresh_tokens[refresh] = form.username

    return TokenPair(access_token=access, refresh_token=refresh)

@app_auth.post("/token/refresh", response_model=Token)
def renovar(body: RefreshRequest):
    username = refresh_tokens.pop(body.refresh_token, None)  # invalida all'uso
    if not username:
        raise HTTPException(401, "Refresh token non valido o già usato")

    usuario = USUARIOS_DB.get(username)
    if not usuario or not usuario["activo"]:
        raise HTTPException(401, "Utente non valido")

    nuevo_access = crear_token({"sub": username}, expira_en=15)
    return Token(access_token=nuevo_access)
```
</details>

---
### Esercizio 2 — Cambio password

**Consegna:** Aggiungi l'endpoint `POST /me/cambiar-password`:

- Richiede autenticazione (Bearer token)
- Body: `{"password_actual": "...", "password_nuevo": "...", "confirmacion": "..."}`
- Validare che `password_actual` sia corretta
- Validare che `password_nuevo == confirmacion`
- Validare che la nuova sia diversa dall'attuale
- Aggiornare l'hash nel DB

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
class CambioPassword(BaseModel):
    password_actual: str = Field(min_length=1)
    password_nuevo:  str = Field(min_length=8)
    confirmacion:    str = Field(min_length=8)

@app_auth.post("/me/cambiar-password")
def cambiar_password(
    datos:   CambioPassword,
    usuario: Usuario = Depends(obtener_usuario_actual)
):
    datos_bd = USUARIOS_DB[usuario.username]

    if not verificar_password(datos.password_actual, datos_bd["password_hash"]):
        raise HTTPException(400, "La password attuale non è corretta")

    if datos.password_nuevo != datos.confirmacion:
        raise HTTPException(400, "La nuova password e la conferma non coincidono")

    if verificar_password(datos.password_nuevo, datos_bd["password_hash"]):
        raise HTTPException(400, "La nuova password deve essere diversa da quella attuale")

    datos_bd["password_hash"] = hash_password(datos.password_nuevo)
    return {"mensaje": "Password aggiornata correttamente"}
```
</details>

---
### Esercizio 3 — Auditing

**Consegna:** Aggiungi un sistema di auditing che registri gli accessi:

- Modello `LogAcceso`: id, username, endpoint, metodo, timestamp, ip (simulata)
- Registra automaticamente ogni request autenticata con un `BackgroundTask`
- `GET /admin/logs` — solo admin, elenca gli ultimi 50 log

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
from fastapi import BackgroundTasks, Request
from datetime import datetime

logs_acceso: list[dict] = []

def registrar_acceso(username: str, endpoint: str, metodo: str):
    logs_acceso.append({
        "id":        len(logs_acceso) + 1,
        "username":  username,
        "endpoint":  endpoint,
        "metodo":    metodo,
        "timestamp": datetime.now().isoformat(),
        "ip":        "127.0.0.1"  # simulata
    })

@app_auth.get("/recursos-auditados")
def recursos_auditados(
    request:    Request,
    bg_tasks:   BackgroundTasks,
    usuario:    Usuario = Depends(obtener_usuario_actual)
):
    bg_tasks.add_task(registrar_acceso, usuario.username, str(request.url.path), request.method)
    return {"datos": "risorse private", "usuario": usuario.nombre}

@app_auth.get("/admin/logs")
def ver_logs(admin: Usuario = Depends(solo_admin)):
    return logs_acceso[-50:]
```
</details>

---
### Esercizio 4 — Proteggere l'API della biblioteca

**Consegna:** Prendi l'API della biblioteca della Lezione 4 e aggiungi l'autenticazione:

- `GET /libros` e `GET /libros/{id}` — qualsiasi utente autenticato
- `POST /libros`, `PATCH /libros/{id}` — solo ruolo `editor` o `admin`
- `DELETE /libros/{id}` — solo `admin`
- `GET /autores` — pubblico (senza autenticazione)
- Il campo `creado_por` in Libro deve registrare lo username del creatore

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
# Dipendenze di ruolo
def rol_editor_o_admin(usuario: Usuario = Depends(obtener_usuario_actual)):
    if usuario.rol not in ("editor", "admin"):
        raise HTTPException(403, "È richiesto il ruolo editor o admin")
    return usuario

def solo_admin_bib(usuario: Usuario = Depends(obtener_usuario_actual)):
    if usuario.rol != "admin":
        raise HTTPException(403, "Solo admin può eliminare libri")
    return usuario

# GET pubblico (senza Depends di auth)
@app.get("/autores")  # già esistente, nessuna modifica
def listar_autores_pub(db: Session = Depends(get_db)):
    return db.query(Autor).all()

# GET autenticato
@app.get("/libros/privado", response_model=list[LibroConAutor])
def listar_auth(db: Session = Depends(get_db), _: Usuario = Depends(obtener_usuario_actual)):
    return db.query(Libro).all()

# POST solo editor/admin — salva il creatore
@app.post("/libros/auth", response_model=LibroConAutor, status_code=201)
def crear_auth(datos: LibroCreate, db: Session = Depends(get_db),
               usuario: Usuario = Depends(rol_editor_o_admin)):
    libro = Libro(**datos.model_dump(), creado_por=usuario.username)  # aggiungere Column al modello ORM
    db.add(libro); db.commit(); db.refresh(libro)
    return libro

# DELETE solo admin
@app.delete("/libros/auth/{id}", status_code=204)
def eliminar_auth(id: int, db: Session = Depends(get_db),
                  _: Usuario = Depends(solo_admin_bib)):
    libro = db.get(Libro, id)
    if not libro: raise HTTPException(404, "Non trovato")
    db.delete(libro); db.commit()
```
</details>

---
### Esercizio 5 (sfida) — API con auth completa

**Consegna:** Costruisci un'API di gestione progetti con autenticazione e autorizzazione complete:

**Ruoli:** `admin`, `manager`, `developer`

**Risorse e permessi:**
- `GET /proyectos` — tutti gli autenticati
- `POST /proyectos` — manager e admin
- `GET /proyectos/{id}/miembros` — tutti
- `POST /proyectos/{id}/miembros` — manager del progetto o admin
- `DELETE /proyectos/{id}` — solo admin

**Regola chiave:** ogni progetto ha un `manager_username`. Solo quel manager (o l'admin) può aggiungere membri al proprio progetto.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from fastapi import FastAPI, HTTPException, Depends
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests

app_proj = FastAPI(title="Gestione Progetti")

# Utenti
USERS = {
    "admin1":  {"rol": "admin",     "hash": hash_password("admin1pass")},
    "mgr1":    {"rol": "manager",   "hash": hash_password("mgr1pass")},
    "dev1":    {"rol": "developer", "hash": hash_password("dev1pass")},
}

# Progetti in memoria
proyectos: dict[int, dict] = {}
miembros:  dict[int, list] = {}   # {proyecto_id: [usernames]}
seq = 0

class ProyectoCreate(BaseModel):
    nombre:      str = Field(min_length=1)
    descripcion: Optional[str] = None

class MiembroAdd(BaseModel):
    username: str

oauth2 = OAuth2PasswordBearer(tokenUrl="/login")

def usuario_actual(token: str = Depends(oauth2)):
    payload  = decodificar_token(token)
    username = payload.get("sub")
    if username not in USERS: raise HTTPException(401, "Token non valido")
    return {"username": username, "rol": USERS[username]["rol"]}

@app_proj.post("/login")
def login_proj(form: OAuth2PasswordRequestForm = Depends()):
    u = USERS.get(form.username)
    if not u or not verificar_password(form.password, u["hash"]):
        raise HTTPException(401, "Credenziali errate")
    return {"access_token": crear_token({"sub": form.username}), "token_type": "bearer"}

@app_proj.get("/proyectos")
def listar(u=Depends(usuario_actual)): return list(proyectos.values())

@app_proj.post("/proyectos", status_code=201)
def crear(datos: ProyectoCreate, u=Depends(usuario_actual)):
    if u["rol"] not in ("manager", "admin"): raise HTTPException(403, "È richiesto manager o admin")
    global seq; seq += 1
    p = {"id": seq, "manager_username": u["username"], **datos.model_dump()}
    proyectos[seq] = p; miembros[seq] = [u["username"]]
    return p

@app_proj.get("/proyectos/{id}/miembros")
def ver_miembros(id: int, u=Depends(usuario_actual)):
    if id not in proyectos: raise HTTPException(404, "Progetto non trovato")
    return miembros.get(id, [])

@app_proj.post("/proyectos/{id}/miembros")
def añadir_miembro(id: int, body: MiembroAdd, u=Depends(usuario_actual)):
    if id not in proyectos: raise HTTPException(404, "Progetto non trovato")
    p = proyectos[id]
    es_manager_del_proyecto = u["username"] == p["manager_username"]
    if u["rol"] != "admin" and not es_manager_del_proyecto:
        raise HTTPException(403, "Solo il manager del progetto o l'admin può aggiungere membri")
    if body.username not in USERS: raise HTTPException(404, "L'utente non esiste")
    if body.username not in miembros[id]:
        miembros[id].append(body.username)
    return {"miembros": miembros[id]}

@app_proj.delete("/proyectos/{id}", status_code=204)
def eliminar(id: int, u=Depends(usuario_actual)):
    if u["rol"] != "admin": raise HTTPException(403, "Solo admin può eliminare")
    if id not in proyectos: raise HTTPException(404, "Progetto non trovato")
    del proyectos[id]; del miembros[id]
```
</details>

---
## Riepilogo

```python
# Hash della password
pwd_context = CryptContext(schemes=["bcrypt"])
hashed = pwd_context.hash("mi_password")
ok     = pwd_context.verify("mi_password", hashed)

# Creare il JWT
token = jwt.encode({"sub": username, "exp": datetime.now(tz) + timedelta(minutes=30)},
                   SECRET_KEY, algorithm="HS256")

# Decodificare il JWT
payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])

# OAuth2 Password Flow
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

@app.post("/token")
def login(form: OAuth2PasswordRequestForm = Depends()):
    # verificare form.username / form.password
    return {"access_token": crear_token({"sub": form.username}), "token_type": "bearer"}

def usuario_actual(token: str = Depends(oauth2_scheme)):
    payload = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
    return payload["sub"]

@app.get("/privado")
def ruta_privada(user = Depends(usuario_actual)):  # richiede un token valido
    return {"usuario": user}
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 6** aggiungiamo il livello professionale: documentazione OpenAPI/Swagger arricchita, CORS per i frontend, middleware di logging e rate limiting per proteggere l'API dagli abusi.